#Statement of Intent
My goal is to build a full ETL pipepline and generate ML models. The models will predict the winner of a match given certain parameters, and they will be evaluated against each other to figure out the best. Deployment of this model could be used for (hopefully) bragging rights or possibly used to bet for the moneyline, though the latter is not the desired purpose.

#Data Sourcing
The data is sourced from match statistics provided by Jeff Sackmann.

The last 4 years worth of matches from Jeff's GitHub repo have been stored in s3 on a free AWS account. I am starting the feature engineering by initiating the spark session and pulling the raw files from the s3 bucket.

In [0]:
from pyspark.sql import (
    SparkSession,
    types,
    functions as F,
)

from pyspark.sql.window import Window

I convert the files to delta for faster pulling, version control, and updating records if needed. I only need to run that cell once.

In [0]:
#Convert to delta
df = spark.read.csv("s3://data-storage-for-projects/Tennis Analytics Project/raw/"
                    , header=True, inferSchema=True)
df.write.format("delta").mode("overwrite").save("s3://data-storage-for-projects/Tennis Analytics Project/delta/")

In [0]:
#Bring data into notebook
df = spark.read.format("delta").load("s3://data-storage-for-projects/Tennis Analytics Project/delta/")
display(df)

#Baseline Model aka V1
For my baseline models, I am using tournament conditions such as surface, tournament level, best of, and round of the match. I will also include rankings of each player, but I will need to have two rows per match so that each player is player 1. The data in its current form has winners all in one column and could train the model incorrectly. As for the types of models, I plan to create and compare three--logistic regression, random forest, and gradient boosting.

In [0]:
#Columns for player a and player b rank, age, and hand. It also says if a won. Two are used to prevent data leakage.
d1 = df.select('surface', 'tourney_level', 'round',
        'best_of', 'winner_hand', 'loser_hand',
        F.col('winner_id').alias('id_a'), F.col('loser_id').alias('id_b'), 
        F.col('winner_rank').alias('id_a_rank'), F.col('loser_rank').alias('id_b_rank'),
        F.col('winner_age').alias('id_a_age'), F.col('loser_age').alias('id_b_age')
        ).withColumn('rank_diff', F.col('id_a_rank') - F.col('id_b_rank')) \
        .withColumn('best_of', F.when(F.col('best_of')==3, 0).otherwise(1)) \
        .withColumn('id_a_hand', F.when(F.col('winner_hand')=='R', 0).otherwise(1)) \
        .withColumn('id_b_hand', F.when(F.col('loser_hand')=='R', 0).otherwise(1))  \
        .withColumn("round_encoded",
        F.when(df.round == "RR", 0)
        .when(df.round == "R128", 1)
        .when(df.round == "R64", 2)
        .when(df.round == "R32", 3)
        .when(df.round == "R16", 4)
        .when(df.round == "QF", 5)
        .when(df.round == "SF", 6)
        .when(df.round == "BR", 6.5)
        .when(df.round == "F", 7)
        .otherwise(None)
        ) \
        .withColumn('id_a_won', F.lit(1))


d2 = df.select('surface', 'tourney_level', 'round',
        'best_of', 'winner_hand', 'loser_hand',
        F.col('loser_id').alias('id_a'), F.col('winner_id').alias('id_b'), 
        F.col('loser_rank').alias('id_a_rank'), F.col('winner_rank').alias('id_b_rank'),
        F.col('loser_age').alias('id_a_age'), F.col('winner_age').alias('id_b_age')
        ).withColumn('rank_diff', F.col('id_a_rank') - F.col('id_b_rank')) \
        .withColumn('best_of', F.when(F.col('best_of')==3, 0).otherwise(1)) \
        .withColumn('id_a_hand', F.when(F.col('loser_hand')=='R', 0).otherwise(1)) \
        .withColumn('id_b_hand', F.when(F.col('winner_hand')=='R', 0).otherwise(1)) \
        .withColumn("round_encoded",
        F.when(F.col('round') == "RR", 0)
        .when(F.col('round') == "R128", 1)
        .when(F.col('round') == "R64", 2)
        .when(F.col('round') == "R32", 3)
        .when(F.col('round') == "R16", 4)
        .when(F.col('round') == "QF", 5)
        .when(F.col('round') == "SF", 6)
        .when(F.col('round') == "BR", 6)
        .when(F.col('round') == "F", 7)
        .otherwise(None)
        ) \
        .withColumn('id_a_won', F.lit(0))

d = d1.union(d2).drop('winner_hand', 'loser_hand', 'round') \
        .filter(F.col('surface').isNotNull() & F.col('rank_diff').isNotNull()
                & F.col('id_a_age').isNotNull() & F.col('id_b_age').isNotNull()) 
display(d)

In [0]:
#Create ML Pipeline starting with training data, string indexer and one hot encoder for logisitic regression
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

#Traininng and testing data
train_df, test_df = d.randomSplit([.8, .2], seed=42)

#Indexing and Encoding
indexer_surface = StringIndexer(inputCol='surface', outputCol='surface_index')
encoder_surface = OneHotEncoder(inputCol='surface_index', outputCol='surface_vec')

indexer_tourney = StringIndexer(inputCol='tourney_level', outputCol='tourney_index')
encoder_tourney = OneHotEncoder(inputCol='tourney_index', outputCol='tourney_vec')

#X variables
feature_cols = ['surface_vec', 'tourney_vec', 'round_encoded', 'best_of', 'id_a_rank',
                'id_b_rank','id_a_age','id_b_age','rank_diff','id_a_hand','id_b_hand']

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')

In [0]:
#This function will be used later to evaluate each model through a variety of metrics
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics

def evaluate_model(predictions, model_name=''):
    # ==========================================
    # 1. AUC Metrics (BinaryClassificationEvaluator)
    # ==========================================
    auc_roc_evaluator = BinaryClassificationEvaluator(labelCol='id_a_won', metricName='areaUnderROC')
    auc_pr_evaluator = BinaryClassificationEvaluator(labelCol='id_a_won', metricName='areaUnderPR')

    auc_roc = auc_roc_evaluator.evaluate(predictions)
    auc_pr = auc_pr_evaluator.evaluate(predictions)

    # ==========================================
    # 2. Accuracy, Precision, Recall, F1 (MulticlassClassificationEvaluator)
    # ==========================================
    accuracy_evaluator = MulticlassClassificationEvaluator(labelCol='id_a_won', predictionCol='prediction', metricName='accuracy')
    precision_evaluator = MulticlassClassificationEvaluator(labelCol='id_a_won', predictionCol='prediction', metricName='weightedPrecision')
    recall_evaluator = MulticlassClassificationEvaluator(labelCol='id_a_won', predictionCol='prediction', metricName='weightedRecall')
    f1_evaluator = MulticlassClassificationEvaluator(labelCol='id_a_won', predictionCol='prediction', metricName='f1')

    accuracy = accuracy_evaluator.evaluate(predictions)
    precision = precision_evaluator.evaluate(predictions)
    recall = recall_evaluator.evaluate(predictions)
    f1 = f1_evaluator.evaluate(predictions)

    # ==========================================
    # 3. Confusion Matrix (manual calculation)
    # ==========================================
    predictions.groupBy('id_a_won', 'prediction').count().show()

    # Or create it properly:
    tp = predictions.filter((F.col('id_a_won') == 1) & (F.col('prediction') == 1)).count()
    tn = predictions.filter((F.col('id_a_won') == 0) & (F.col('prediction') == 0)).count()
    fp = predictions.filter((F.col('id_a_won') == 0) & (F.col('prediction') == 1)).count()
    fn = predictions.filter((F.col('id_a_won') == 1) & (F.col('prediction') == 0)).count()

    print("=" * 60)
    print(f"{model_name.upper()} - EVALUATION METRICS")
    print("=" * 60)
    print(f"AUC-ROC:              {auc_roc:.4f}")
    print(f"AUC-PR:               {auc_pr:.4f}")
    print(f"Accuracy:             {accuracy:.4f}")
    print(f"Precision (Weighted): {precision:.4f}")
    print(f"Recall (Weighted):    {recall:.4f}")
    print(f"F1 Score:             {f1:.4f}")
    print("=" * 60)
    print("CONFUSION MATRIX")
    print("=" * 60)
    print(f"True Positives:       {tp:,}")
    print(f"True Negatives:       {tn:,}")
    print(f"False Positives:      {fp:,}")
    print(f"False Negatives:      {fn:,}")
    print("=" * 60)

    return {
        'auc_roc': auc_roc,
        'auc_pr': auc_pr,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': {'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}
    }

In [0]:
#Logistic Regression Model
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol='features', labelCol='id_a_won')

pipeline_lr = Pipeline(stages = [
    indexer_surface, encoder_surface,
    indexer_tourney, encoder_tourney,
    assembler,
    lr
])

#Create Model and save to s3
lr_model = pipeline_lr.fit(train_df)
lr_model.save("s3://data-storage-for-projects/Tennis Analytics Project/v1-models/logistic-regressions/1.0/")

#Generate Predictions and Evaluate Accuracy
lr_predict = lr_model.transform(test_df)
lr_eval = evaluate_model(lr_predict, 'Logistic Regression')

#Remove from Databricks Memory to create more models
del lr_model, lr_predict

In [0]:
#Random Forest
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol='features', labelCol='id_a_won', numTrees=50, maxDepth=10)

feature_cols = ['surface_index', 'tourney_index', 'round_encoded', 'best_of', 'id_a_rank',
                'id_b_rank','id_a_age','id_b_age','rank_diff','id_a_hand','id_b_hand']

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')

pipeline_rf = Pipeline(stages = [
    indexer_surface,
    indexer_tourney,
    assembler,
    rf
])

#Create Model and save to s3
rf_model = pipeline_rf.fit(train_df)
rf_model.save("s3://data-storage-for-projects/Tennis Analytics Project/v1-models/random-forests/1.1/")

#Generate Predictions and Evaluate Accuracy
rf_predict = rf_model.transform(test_df)
rf_eval = evaluate_model(rf_predict, 'Random Forest')

#Remove from Databricks Memory to create more models
del rf_model, rf_predict

In [0]:
#Gradient Boosting
from pyspark.ml.classification import GBTClassifier

gb = GBTClassifier(labelCol="id_a_won", featuresCol="features", maxDepth=6, maxIter=20)

feature_cols = ['surface_index', 'tourney_index', 'round_encoded', 'best_of', 'id_a_rank',
                'id_b_rank','id_a_age','id_b_age','rank_diff','id_a_hand','id_b_hand']

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')

pipeline_gb = Pipeline(stages = [
    indexer_surface,
    indexer_tourney,
    assembler,
    gb
])

#Create Model and save to s3
gb_model = pipeline_gb.fit(train_df)
gb_model.save("s3://data-storage-for-projects/Tennis Analytics Project/v1-models/gradient-boostings/1.0/")

#Generate Predictions and Evaluate Accuracy
gb_predict = gb_model.transform(test_df)
gb_eval = evaluate_model(gb_predict, 'Gradient Boosting')

#Remove from Databricks Memory to create more models
del gb_model, gb_predict